In [2]:
# Importar librerías
import numpy as np
import torch
from torch import nn
from torch import optim
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from collections import OrderedDict

# Configuración para gráficos
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

print("Librerías importadas correctamente")

Librerías importadas correctamente


In [3]:

# Definir una transformación para normalizar los datos
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Descargar y cargar los datos de entrenamiento
trainset = datasets.MNIST('MNIST_data/', download=True, train=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

print("Dataset descargado correctamente")

Dataset descargado correctamente


In [4]:
# Hiperparámetros
input_size = 784
hidden_sizes = [400, 200, 100]
output_size = 10

# Construir la red
model = nn.Sequential(OrderedDict([
    ('fc1', nn.Linear(input_size, hidden_sizes[0])),
    ('relu1', nn.ReLU()),
    ('fc2', nn.Linear(hidden_sizes[0], hidden_sizes[1])),
    ('relu2', nn.ReLU()),
    ('fc3', nn.Linear(hidden_sizes[1], hidden_sizes[2])),
    ('relu3', nn.ReLU()),
    ('logits', nn.Linear(hidden_sizes[2], output_size))
]))

print("Red construida:")
print(model)

Red construida:
Sequential(
  (fc1): Linear(in_features=784, out_features=400, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=400, out_features=200, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=200, out_features=100, bias=True)
  (relu3): ReLU()
  (logits): Linear(in_features=100, out_features=10, bias=True)
)


In [5]:
# Ahora seguimos con la parte del entreamiento
# Definiendo la funcion de perdida y el optimizador
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=0.01)

print("Función de pérdida definida: CrossEntropyLoss")
print("Optimizador definido: SGD con lr=0.01")

Función de pérdida definida: CrossEntropyLoss
Optimizador definido: SGD con lr=0.01


In [6]:
print('Pesos iniciales - ', model.fc1.weight)

print('Pesos iniciales')
images, labels = next(iter(trainloader))

images.resize_(64, 784) # Cada imagen es 28x28 como una matriz lo que hace es convertirlo en un vectory, cada valor de los 784 es un pixel, un color

optimizer.zero_grad() # Borra los gradientes anteriores para que no se acumulen

# Pase hacia adelante, luego hacia atrás, luego actualizar pesos
output = model.forward(images)
loss = criterion(output, labels)
loss.backward()
print('Gradiente - ', model.fc1.weight.grad)
optimizer.step()

print('Pesos actualizados - ', model.fc1.weight)

Pesos iniciales -  Parameter containing:
tensor([[-0.0035,  0.0128, -0.0270,  ...,  0.0324,  0.0212,  0.0198],
        [ 0.0022, -0.0198,  0.0166,  ...,  0.0137,  0.0079,  0.0126],
        [-0.0308, -0.0034, -0.0266,  ..., -0.0217, -0.0331, -0.0250],
        ...,
        [-0.0206,  0.0290,  0.0282,  ...,  0.0048, -0.0230,  0.0125],
        [ 0.0129, -0.0145,  0.0165,  ...,  0.0164,  0.0023, -0.0356],
        [ 0.0128, -0.0198,  0.0024,  ...,  0.0042, -0.0069, -0.0243]],
       requires_grad=True)
Pesos iniciales
Gradiente -  tensor([[ 3.6080e-04,  3.6080e-04,  3.6080e-04,  ...,  3.6080e-04,
          3.6080e-04,  3.6080e-04],
        [ 9.4666e-05,  9.4666e-05,  9.4666e-05,  ...,  9.4666e-05,
          9.4666e-05,  9.4666e-05],
        [-6.7168e-04, -6.7168e-04, -6.7168e-04,  ..., -6.7168e-04,
         -6.7168e-04, -6.7168e-04],
        ...,
        [-3.9037e-04, -3.9037e-04, -3.9037e-04,  ..., -3.9037e-04,
         -3.9037e-04, -3.9037e-04],
        [ 8.1506e-04,  8.1506e-04,  8.1506e-

In [9]:
# Ahora seguimos con el entrenamiento de vdd
optimizer = optim.SGD(model.parameters(), lr=0.003) # Para ajustar los pesos y reducir el error

epochs = 3 # La cantidad de veces que el entrenamiento pasara por el dataset
print_every = 40 # Mostrara la perdida cada 40 pasos para no imprimir demasiadas veces
steps = 0 # Contador para saber cuantos pasos hemos dado en total

for e in range(epochs):  # Bucle que se repite 3 veces
    running_loss = 0
    for images, labels in iter(trainloader): # Bucle que itera por cada batch de 64 imagenes
        steps += 1 # Contador de pasos
        
        # Aplanar imágenes MNIST en un vector de 784
        images.resize_(images.size()[0], 784)
        
        # Poner a cero los gradientes
        optimizer.zero_grad()
        
        # Pases hacia adelante y hacia atrás
        output = model.forward(images)
        loss = criterion(output, labels) # Calcula el costo y error
        loss.backward() # Calcula los gradientes para cada peso
        optimizer.step()
        
        running_loss += loss.item()
        
        if steps % print_every == 0:
            print("Época: {}/{}... ".format(e+1, epochs),
                  "Pérdida: {:.4f}".format(running_loss/print_every))
            running_loss = 0

Época: 1/3...  Pérdida: 2.2994
Época: 1/3...  Pérdida: 2.2956
Época: 1/3...  Pérdida: 2.2899
Época: 1/3...  Pérdida: 2.2859
Época: 1/3...  Pérdida: 2.2796
Época: 1/3...  Pérdida: 2.2789
Época: 1/3...  Pérdida: 2.2708
Época: 1/3...  Pérdida: 2.2653
Época: 1/3...  Pérdida: 2.2563
Época: 1/3...  Pérdida: 2.2564
Época: 1/3...  Pérdida: 2.2508
Época: 1/3...  Pérdida: 2.2417
Época: 1/3...  Pérdida: 2.2332
Época: 1/3...  Pérdida: 2.2272
Época: 1/3...  Pérdida: 2.2129
Época: 1/3...  Pérdida: 2.2023
Época: 1/3...  Pérdida: 2.1965
Época: 1/3...  Pérdida: 2.1768
Época: 1/3...  Pérdida: 2.1617
Época: 1/3...  Pérdida: 2.1474
Época: 1/3...  Pérdida: 2.1342
Época: 1/3...  Pérdida: 2.1113
Época: 1/3...  Pérdida: 2.0967
Época: 2/3...  Pérdida: 1.1357
Época: 2/3...  Pérdida: 2.0364
Época: 2/3...  Pérdida: 2.0045
Época: 2/3...  Pérdida: 1.9767
Época: 2/3...  Pérdida: 1.9447
Época: 2/3...  Pérdida: 1.8951
Época: 2/3...  Pérdida: 1.8578
Época: 2/3...  Pérdida: 1.8357
Época: 2/3...  Pérdida: 1.7689
Época: 2

In [ ]:
# Calcular precisión
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

correct = 0
total = 0

with torch.no_grad():
    for images, labels in trainloader:
        images = images.resize_(images.size()[0], 784)
        
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Precisión: {accuracy:.2f}%")

In [60]:
# En esta parte nos daremos si realmente nuestra red aprendio de forma correcta o no
images, labels = next(iter(trainloader))

img = images[0].view(1, 784)

# Desactivar gradientes para acelerar esta parte
with torch.no_grad():
    logits = model.forward(img)

# La salida de la red son logits, necesitamos aplicar softmax para probabilidades
ps = F.softmax(logits, dim=1)

print("Predicción de la red:")
print(ps)
print("\nDígito predicho:", torch.argmax(ps))
print("Dígito real:", labels[0].item())

Predicción de la red:
tensor([[1.2484e-03, 1.0920e-06, 1.1441e-02, 9.7138e-06, 3.2881e-03, 7.9969e-04,
         9.8186e-01, 1.2213e-06, 1.2737e-03, 7.7971e-05]])

Dígito predicho: tensor(6)
Dígito real: 6
